In [ ]:
# !pip install --upgrade requirements.txt

In [ ]:
import torch
import os
import warnings
from typing import Dict, Any, Optional, Tuple

# suppress warnings
warnings.filterwarnings('ignore')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")

Model Configuration

In [ ]:
from dataclasses import dataclass


@dataclass
class ModelConfig:
    '''Model specific configuration settings'''
    model_name: str
    pad_token: str
    pad_token_id: int
    padding_side: str
    eos_token: str
    eos_token_id: int
    vocab_size: int
    model_type: str         # model architecture type

@dataclass
class TrainingConfig:
    '''Training hyperparameters'''
    output_dir: str
    batch_size: int = 16
    gradient_accumulation_steps: int = 8    # Steps to accumulte gradient
    learning_rate: float = 1e-4
    max_steps: int = 1000
    max_seq_length: int = 2048
    lora_r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.05
    save_steps: int = 250           # steps between saving checkpoint
    logging_steps: int = 10          # steps between logging outputs
    warmup_ratio: float = 0.1       # warm-steps for learning rate


Model Configuration

In [ ]:
from transformers import AutoTokenizer, AutoConfig

def auto_configure_model(model_name: str, custom_pad_token: str = None) -> ModelConfig:
    '''
    Automatically configure any model by extracting information from it's tokenizer'''
    print(f'Loading model config: {model_name}')

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model_config = AutoConfig.from_pretrained(model_name)

    # extract model infor
    model_type = getattr(model_config, 'model_type', 'unknown')
    vocab_size = getattr(model_config, 'vocab_size', len(tokenizer.get_vocab()))

    print(f'Model: {model_type}, vocab_size: {vocab_size:,}')

    eos_token = tokenizer.eos_token
    eos_token_id = tokenizer.eos_token_id

    if eos_token is None:
        raise ValueError(f"Model {model_name} missing EOS token")

    pad_token = tokenizer.pad_token
    pad_token_id = tokenizer.pad_token_id

    if not pad_token:
        if not custom_pad_token:
            raise ValueError(f"Model needs custom_pad_token. Use '<|eot_id|>' for Llama, '<|im_end|>' for Qwen")

        pad_token = custom_pad_token
        if pad_token in tokenizer.get_vocab():
            pad_token_id = tokenizer.get_vocab()[pad_token]
        else:
            tokenizer.add_special_token({'pad_token': pad_token})
            pad_token_id = tokenizer.pad_token_id

    print(f"Configured - pad: '{pad_token}' (ID: {pad_token_id}), eos: '{eos_token}', (ID: {eos_token_id})")

    return ModelConfig(
        model_name=model_name,
        pad_token=pad_token,
        pad_token_id=pad_token_id,
        padding_side='Left',
        eos_token=eos_token,
        eos_token_id=eos_token_id,
        vocab_size=vocab_size,
        model_type=model_type
    )

# auto_configure_model("Qwen/Qwen3-0.6B")

In [ ]:
def create_training_configuration(model_name: str, **kwargs) -> TrainingConfig:
    '''Create training configuration'''
    model_name_only = model_name.split('/')[-1].replace('-', '_').replace('.', '_')
    default_output_dir = f"./{model_name_only}_xlam"

    config_dict = {'output_dir': default_output_dir, **kwargs}
    return TrainingConfig(**config_dict)

# test the function
# create_training_configuration("Qwen/Qwen3-0.6B", batch_size=4)

In [ ]:
def setup_hardware_config() -> tuple[torch.dtype, str]:
    """Choose a precision supported by the active CUDA device."""

    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU")

    print(f"GPU: {torch.cuda.get_device_name(0)}")

    if torch.cuda.is_bf16_supported():
        compute_dtype = torch.bfloat16
        print("Using bfloat16 mixed precision")
    else:
        compute_dtype = torch.float16
        print("Using float16 mixed precision")

    # SDPA is built into modern PyTorch and does not require FlashAttention.
    attn_implementation = "sdpa"

    return compute_dtype, attn_implementation


compute_dtype, attn_implementation = setup_hardware_config()

In [ ]:
from transformers import AutoTokenizer

# Tokenizer setup function
def setup_tokenizer(model_config: ModelConfig) -> AutoTokenizer:
    '''Initialize and configure the tokenizer using model configuration'''
    print(f"Loading tokenizer for {model_config.model_name}")

    tokenizer =AutoTokenizer.from_pretrained(model_config.model_name, use_fast=None)

    # Configure padding token using values from model_config
    tokenizer.pad_token = model_config.pad_token
    tokenizer.pad_token_id = model_config.pad_token_id
    tokenizer.padding_side = model_config.padding_side

    print(f"Tokenizer configured - pad: '{model_config.pad_token}' (ID: {model_config.pad_token_id})")

    return tokenizer

In [ ]:
# Data Processing
import json
import multiprocessing
from datasets import load_dataset, Dataset

# def process_xlam_sample(row: Dict[str, Any], tokenizer) -> Dict[str, str]:
#     '''Process a simle xLAM dataset sample into training format
    
#     <user>[user query]</user>

#     <tools>
#     [tool definitions]
#     </tools>

#     <calls>
#     [expected function calls]
#     </calls>[EOS_TOKEN]
#     '''
#     # Format user query
#     formatted_query = f"<user>{row['query']}</user>\n\n"

#     # parse and format available tools
#     try:
#         parsed_tools = json.loads(row['tools'])
#         tools_text = '\n'.join(str(tool) for tool in parsed_tools)
#     except json.JSONDecodeError:
#         tools_text = str(row['tools'])

#     formatted_tools = f"<tools>{tools_text}</tools>\n\n"

#     # Parse and format expected functions tools
#     try:
#         parsed_answers = json.loads(row['answers'])
#         answers_text = '\n'.join(str(answer) for answer in parsed_answers)
#     except json.JSONDecodeError:
#         answers_text = str(row['answers'])

#     formatted_answers = f"<calls>{answers_text}</calls>"

#     # combine all parts with EOD token
#     complete_text = formatted_query + formatted_tools + formatted_answers + tokenizer.eos_token

#     # Update row with processed data
#     row['query'] = formatted_query
#     row['tools'] = formatted_tools
#     row['answers'] = formatted_answers
#     row['text'] = complete_text

#     return row

def process_xlam_sample(row, tokenizer):
    formatted_query = f"<user>{row['query']}</user>\n\n"

    try:
        parsed_tools = json.loads(row["tools"])
    except json.JSONDecodeError as exc:
        raise ValueError("Invalid JSON in the source tools field") from exc

    try:
        parsed_answers = json.loads(row["answers"])
    except json.JSONDecodeError as exc:
        raise ValueError("Invalid JSON in the source answers field") from exc

    tools_text = json.dumps(
        parsed_tools,
        ensure_ascii=False,
        separators=(",", ":"),
    )
    answers_text = json.dumps(
        parsed_answers,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    complete_text = (
        f"{formatted_query}"
        f"<tools>{tools_text}</tools>\n\n"
        f"<calls>{answers_text}</calls>"
        f"{tokenizer.eos_token}"
    )

    return {"text": complete_text}

In [ ]:
def load_and_process_xlam_dataset(tokenizer: AutoTokenizer, sample_size: Optional[int] = None, split: str = None) -> Dataset:
    '''Load and process the complete xLAM dataset for function calling training'''
    
    print("Loading xLAM dataset ...")

    dataset = load_dataset('json', data_files='data/xlam_function_calling_60k.json')
    split_dataset = dataset['train'].train_test_split(test_size=0.3, seed=42)
    train_dataset = split_dataset['train']
    test_dataset = split_dataset['test']

    # dataset = load_dataset("Salesforce/xlam-function-calling-60k", split='train')

    print(f"original dataset size: {len(dataset):,} samples")

    print("Processing into training format...")

    def process_batch(batch):
        processed_texts = []

        for i in range(len(batch["query"])):
            row = {
                "query": batch["query"][i],
                "tools": batch["tools"][i],
                "answers": batch["answers"][i],
            }

            processed_row = process_xlam_sample(row, tokenizer)
            processed_texts.append(processed_row["text"])

        return {"text": processed_texts}
    
    # Process the dataset
    if split:
        if split == 'train': 
            if sample_size and sample_size < len(train_dataset):
                train_dataset = train_dataset.select(range(sample_size))
                print(f"Using sample size: {sample_size} samples")   
            processed_dataset = train_dataset.map(
                process_batch,
                batched=True,
                batch_size=1000,
                num_proc=min(4, multiprocessing.cpu_count()),
                desc='Processing xLAM samples'
            )
        elif split == 'test' or split == 'val':
            if sample_size and sample_size < len(test_dataset):
                test_dataset = test_dataset.select(range(sample_size))
                print(f"Using sample size: {sample_size} samples")   
            processed_dataset = test_dataset.map(
                process_batch,
                batched=True,
                batch_size=1000,
                num_proc=min(4, multiprocessing.cpu_count()),
                desc='Processing xLAM samples'
            )
    else:
        if sample_size and sample_size < len(dataset):
            dataset = dataset.select(range(sample_size))
            print(f"Using sample size: {sample_size} samples")   
        processed_dataset = dataset.map(
            process_batch,
            batched=True,
            batch_size=1000,
            num_proc=min(4, multiprocessing.cpu_count()),
            desc='Processing xLAM samples'
        )

    print("Dataset processing complete")
    print(f"Final dataset size: {len(processed_dataset):,} samples")
    print(f"Average text length: {sum(len(text) for text in processed_dataset['text']) / len(processed_dataset):,.0f} characters")

    return processed_dataset

In [ ]:
def preview_dataset_sample(dataset: Dataset, index: int=0) -> None:
    '''Display a formatted preview of a dataset sample inspection'''
    if index >= len(dataset):
        print(f"index {index} is out of range.")
        return
    sample = dataset[index]

    print(f"Dataset sample preview: index: {index}")
    print("=" * 80)

    print("\n Raw Components:")
    print(f"Query: {sample['query'][:200]}{'...' if len(sample['query']) > 200 else ''}")
    print(f"Tools: {sample['tools'][:200]}{'...' if len(sample['tools']) > 200 else ''}")
    print(f"Answers: {sample['answers'][:200]}{'...' if len(sample['answers']) > 200 else ''}")
    
    print(f"\n📝 Complete Training Text:")
    print("-" * 40)
    print(sample['text'])
    print("-" * 40)
    
    print(f"\n📊 Sample Statistics:")
    print(f"   • Text length: {len(sample['text']):,} characters")
    print(f"   • Estimated tokens: ~{len(sample['text']) // 4:,} tokens")
    
    print("\n✅ Preview complete!")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

def create_qlora_model(model_config: ModelConfig,
                       tokenizer: AutoTokenizer,
                       compute_dtype: torch.dtype,
                       attn_implementation: str) -> AutoModelForCausalLM:
    '''Create and configure a QLoRA model for efficient fine-tuning'''
    print(f"Create QLoRA model: {model_config.model_name}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading quantized model")

    model = AutoModelForCausalLM.from_pretrained(
        model_config.model_name,
        quantization_config=bnb_config,
        device_map={"": 0},
        attn_implementation=attn_implementation,
        dtype=compute_dtype,
        trust_remote_code=True
    )

    model = prepare_model_for_kbit_training(
        model,
        gradient_checkpointing_kwargs={'use_reentrant': True}
    )

    # configure tokenizer setting in model
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False # Disable cache for training

    print("QLoRA model prepared successfully")
    print("Model memory footprint: ~{model.get_memory_footprint() / 1e9:.1f}")
    
    return model

In [ ]:
# QLoRA Training setup
def create_lora_config(training_config: TrainingConfig) -> LoraConfig:
    target_modules = [
        'k_proj', 'q_proj', 'v_proj', 'o_proj', # Attention projection
        'gate_proj', 'down_proj', 'up_proj' # Feed-forward projections
    ]

    lora_config = LoraConfig(
        lora_alpha=training_config.lora_alpha,
        lora_dropout=training_config.lora_dropout,
        r=training_config.lora_r,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=target_modules
    )
    return lora_config

In [ ]:
# Training Execution
def train_qlora_model(dataset: Dataset,
                      model: AutoModelForCausalLM,
                      training_config: TrainingConfig,
                      compute_dtype: torch.dtype) -> SFTTrainer:

    peft_config = create_lora_config(training_config)
    
    use_bf16 = compute_dtype == torch.bfloat16
    use_fp16 = compute_dtype == torch.float16

    # configure training arguments
    training_arguments = SFTConfig(
        output_dir=training_config.output_dir,
        optim="adamw_8bit",
        per_device_train_batch_size=training_config.batch_size,
        gradient_accumulation_steps=(
            training_config.gradient_accumulation_steps
        ),
        learning_rate=training_config.learning_rate,
        max_steps=training_config.max_steps,
        warmup_steps=int(
            training_config.max_steps * training_config.warmup_ratio
        ),
        lr_scheduler_type="linear",
        logging_steps=training_config.logging_steps,
        save_steps=training_config.save_steps,
        dataset_text_field="text",
        max_length=training_config.max_seq_length,

        # Exactly one should be enabled.
        fp16=use_fp16,
        bf16=use_bf16,

        remove_unused_columns=False,
        dataloader_drop_last=True,
        gradient_checkpointing=True,
        save_total_limit=3,
        load_best_model_at_end=False,
    )

    # Create trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        args=training_arguments
    )

    return trainer


In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
# custom_pad_token = "<|eot_id|>"
custom_pad_token = "<|im_end|>"

model_config = auto_configure_model(MODEL_NAME, custom_pad_token=custom_pad_token)
training_config = create_training_configuration(MODEL_NAME)

print(f"Starting finetuning of: {MODEL_NAME}")
print(f"Model Type: {model_config.model_type}")
print(f"Vocabulary size: {model_config.vocab_size}")
print(f"Pad Token: '{model_config.pad_token}' (ID: {model_config.pad_token_id})")
print(f"Output dir: {training_config.output_dir}")

In [ ]:
print(f"Setting up tokenizer ...")
tokenizer = setup_tokenizer(model_config)

print("\n Loading and processing xLAM dataset ...")
dataset = load_and_process_xlam_dataset(tokenizer, sample_size=None)

print("\n Dataset sample preview")
preview_dataset_sample(dataset, index=0)

print(f"\n Creating QLoRA model ...")
model = create_qlora_model(
    model_config,
    tokenizer,
    compute_dtype,
    attn_implementation
)

print(f"\n Starting training")
trainer = train_qlora_model(
    dataset=dataset,
    model=model,
    training_config=training_config,
    compute_dtype=compute_dtype
)

# Train the model
# trainer.train()

print(f" Finetuning completed for {model_config.model_name.split('/')[-1]}!")
print(f"Model saved to: {training_config.output_dir}")


### Test the Model

In [ ]:
print("\n Loading and processing xLAM dataset ...")
test_dataset = load_and_process_xlam_dataset(tokenizer, sample_size=None, split='test')

print("\n Dataset sample preview")
preview_dataset_sample(test_dataset, index=0)

In [ ]:
# Import required components for inference
from peft import PeftModel

def load_trained_model(model_config: ModelConfig, 
                       adapter_path: str,
                       compute_dtype: torch.dtype,
                       attn_implementation: str) -> Tuple[AutoModelForCausalLM, AutoTokenizer]:
    """
    Load a trained model with LoRA adapters for inference.
    
    This function loads the base model with quantization and applies the trained
    LoRA adapters for efficient inference. It's designed to work after training
    completion or for loading previously saved models.
    
    Args:
        model_config (ModelConfig): Configuration for the base model
        adapter_path (str): Path to the saved LoRA adapter
        compute_dtype (torch.dtype): Computation data type
        attn_implementation (str): Attention implementation
        
    Returns:
        Tuple[AutoModelForCausalLM, AutoTokenizer]: Loaded model and tokenizer
        
    Note:
        You may need to restart the notebook to free GPU memory before loading
        the model for inference, especially after training.
    """
    print(f"🔄 Loading trained model from {adapter_path}")
    
    # Configure quantization for inference
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    
    # Load tokenizer with proper configuration
    tokenizer = setup_tokenizer(model_config)
    print(f"🔤 Tokenizer loaded for {model_config.model_name}")
    
    # Load base model
    print(f"📦 Loading base model {model_config.model_name}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_config.model_name,
        quantization_config=quantization_config,
        torch_dtype=compute_dtype,
        device_map={"": 0},
        attn_implementation=attn_implementation,
        trust_remote_code=True,
    )
    
    # Load LoRA adapters
    print(f"🔗 Loading LoRA adapters from {adapter_path}...")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    
    # Enable evaluation mode
    model.eval()
    
    print("✅ Model loaded successfully and ready for inference!")
    print(f"💾 Total memory usage: ~{model.get_memory_footprint() / 1e9:.1f} GB")
    
    return model, tokenizer

In [ ]:
def generate_function_call(model: AutoModelForCausalLM,
                          tokenizer: AutoTokenizer, 
                          prompt: str,
                          max_new_tokens: int = 512,
                          temperature: float = 0.7,
                          do_sample: bool = True) -> str:
    """
    Generate a function call response using the fine-tuned model.
    
    Args:
        model (AutoModelForCausalLM): Fine-tuned model with LoRA adapters
        tokenizer (AutoTokenizer): Model tokenizer
        prompt (str): Input prompt for function calling
        max_new_tokens (int): Maximum tokens to generate
        temperature (float): Sampling temperature (only used when do_sample=True)
        do_sample (bool): Whether to use sampling
        
    Returns:
        str: Generated response with function calls
        
    Example Prompt Format:
        "<user>Check if the numbers 8 and 1233 are powers of two.</user>\n\n<tools>"
    """
    print(f"🎯 Generating response for prompt...")
    print(f"📝 Input: {prompt}")

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate response with proper parameter handling
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "do_sample": do_sample,
    }
    
    # Only add sampling parameters if do_sample=True
    if do_sample:
        generation_kwargs["temperature"] = temperature
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs
        )
    
    # Decode result
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("✅ Generation completed!")
    print(f"📊 Generated {len(outputs[0]) - len(inputs['input_ids'][0])} new tokens")
    
    return result

In [ ]:
def test_function_calling_examples(model: AutoModelForCausalLM,
                                   tokenizer: AutoTokenizer,
                                   test_dataset: Dataset) -> list[str]:
    """
    Test the model with various function calling examples.
    
    Args:
        model: Fine-tuned model.
        tokenizer: Model tokenizer.
        test_dataset: Dataset rows containing formatted `text` examples.

    Returns:
        Generated responses aligned positionally with test_dataset.
    """
    print("🧪 Testing function calling capabilities...")
    
    results = []
    for i, test_case in enumerate(test_dataset):
        formatted_text = test_case.get("text")
        if not isinstance(formatted_text, str) or "<calls>" not in formatted_text:
            raise ValueError(f"test_dataset[{i}] has no formatted <calls> block")

        # Exclude the expected answer so the model cannot see the ground truth.
        prompt = formatted_text.split("<calls>", maxsplit=1)[0]

        print(f"\n{'='*60}")
        print(f"Test Case {i}: {test_case.get('query', 'dataset example')}")
        print(f"{'='*60}")
        
        result = generate_function_call(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=512,  # Adjust as needed
            temperature=0.7,
            do_sample=True  # Fixed: Use sampling with temperature
        )
        results.append(result)
        print(f"\n🔍 Complete Response:")
        print("-" * 40)
        print(result)
        print("-" * 40)
    
    print("\n✅ All test cases completed!")
    return results

In [ ]:
sample = test_dataset.select(range(2))
preview_dataset_sample(sample)

In [ ]:
# Load and test the trained model
# Note: You may need to restart the notebook to free memory before running this

print("🔄 Loading trained model for testing...")
print("⚠️  If you encounter memory issues, restart the notebook and run only this cell")

# Determine the adapter path based on the training configuration
adapter_path = f"{training_config.output_dir}/checkpoint-{training_config.max_steps}"

print(f"📁 Looking for adapter at: {adapter_path}")

# # Load the trained model
# trained_model, trained_tokenizer = load_trained_model(
#     model_config=model_config,
#     adapter_path=adapter_path,
#     compute_dtype=compute_dtype,
#     attn_implementation=attn_implementation
# )

# Test with a single example
# test_prompt = "<user>Check if the numbers 8 and 1233 are powers of two.</user>\n\n<tools>"
# result = generate_function_call(trained_model, trained_tokenizer, test_prompt)

results = test_function_calling_examples(trained_model, trained_tokenizer, test_dataset)

print(f"\n🎯 Test Result for {model_config.model_name.split('/')[-1]}:")
print("="*80)
print(results)
print("="*80)

In [ ]:
print(len(results))
print(results[0])

In [ ]:
print(results[0])

In [ ]:
import csv
import json
import re
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any


_CALLS_PATTERN = re.compile(r"<calls\b[^>]*>(.*?)</calls>", re.DOTALL | re.IGNORECASE)
_USER_PATTERN = re.compile(r"<user\b[^>]*>(.*?)</user>", re.DOTALL | re.IGNORECASE)


def _normalize_tool_call(call: Any, *, source: str) -> dict[str, Any]:
    """Normalize a tool call to {'name': str, 'arguments': dict}."""
    if not isinstance(call, Mapping):
        raise ValueError(f"{source} must be a JSON object, got {type(call).__name__}")

    # Also accept the OpenAI-style {'function': {'name': ..., 'arguments': ...}} shape.
    function = call.get("function")
    if isinstance(function, Mapping):
        call = function

    # Some generated examples use 'method'; normalize it to the required 'name'.
    name = call.get("name", call.get("method"))
    if not isinstance(name, str) or not name.strip():
        raise ValueError(f"{source} is missing a non-empty tool name: {call!r}")

    arguments = call.get("arguments", {})
    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{source} has invalid JSON arguments: {arguments!r}") from exc

    if not isinstance(arguments, Mapping):
        raise ValueError(f"{source} arguments must be a JSON object: {arguments!r}")

    return {
        "name": name.strip(),
        "arguments": dict(arguments),
    }


def _parse_tool_calls(value: Any, *, source: str, require_tags: bool = False) -> list[dict[str, Any]]:
    """Parse one or more JSON tool calls from raw JSON or <calls> blocks."""
    payloads: list[Any] = []

    if isinstance(value, str):
        call_blocks = _CALLS_PATTERN.findall(value)
        if require_tags and not call_blocks:
            raise ValueError(f"{source} contains no <calls>...</calls> block")

        serialized_payloads = call_blocks or [value]
        for block_index, serialized in enumerate(serialized_payloads):
            try:
                payloads.append(json.loads(serialized.strip()))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{source} call block {block_index} is not valid JSON: {serialized!r}"
                ) from exc
    else:
        payloads.append(value)

    normalized_calls: list[dict[str, Any]] = []
    for payload_index, payload in enumerate(payloads):
        calls = payload if isinstance(payload, list) else [payload]
        for call_index, call in enumerate(calls):
            normalized_calls.append(
                _normalize_tool_call(
                    call,
                    source=f"{source} payload {payload_index}, call {call_index}",
                )
            )

    return normalized_calls


def compare_tool_calls(
    predicted_tool_calls: Sequence[Mapping[str, Any]],
    actual_tool_calls: Sequence[Mapping[str, Any]],
    *,
    prediction_parse_error: str | None = None,
) -> dict[str, bool]:
    """Compare ordered tool names and arguments for one prediction."""
    if prediction_parse_error is not None:
        return {
            "tool_calls_match": False,
            "arguments_match": False,
            "exact_match": False,
        }

    predicted_names = [call["name"] for call in predicted_tool_calls]
    actual_names = [call["name"] for call in actual_tool_calls]
    predicted_arguments = [call["arguments"] for call in predicted_tool_calls]
    actual_arguments = [call["arguments"] for call in actual_tool_calls]

    tool_calls_match = predicted_names == actual_names
    arguments_match = predicted_arguments == actual_arguments
    return {
        "tool_calls_match": tool_calls_match,
        "arguments_match": arguments_match,
        "exact_match": tool_calls_match and arguments_match,
    }


def calculate_function_call_accuracy(
    evaluation_records: Sequence[Mapping[str, Any]],
) -> dict[str, int | float]:
    """Calculate tool-name, argument, and exact-match accuracy."""
    if not evaluation_records:
        raise ValueError("evaluation_records must contain at least one record")

    total = len(evaluation_records)
    tool_call_matches = sum(bool(record["tool_calls_match"]) for record in evaluation_records)
    argument_matches = sum(bool(record["arguments_match"]) for record in evaluation_records)
    exact_matches = sum(bool(record["exact_match"]) for record in evaluation_records)
    return {
        "total_examples": total,
        "tool_call_matches": tool_call_matches,
        "tool_call_accuracy": tool_call_matches / total,
        "argument_matches": argument_matches,
        "argument_accuracy": argument_matches / total,
        "exact_matches": exact_matches,
        "exact_match_accuracy": exact_matches / total,
    }


def evaluate_function_call_predictions(
    test_dataset: Dataset,
    prediction_results: Sequence[str],
) -> list[dict[str, Any]]:
    """Pair prompts and ground-truth calls with calls generated by the model.

    The dataset and prediction list must be positionally aligned. Ground truth
    is read from the `answers` column when present, otherwise from the dataset's
    formatted `text` column. Predictions are parsed from every <calls> block.
    """
    if isinstance(prediction_results, (str, bytes)):
        raise TypeError("prediction_results must be a sequence of response strings")

    if len(test_dataset) != len(prediction_results):
        raise ValueError(
            "test_dataset and prediction_results must have the same length: "
            f"{len(test_dataset)} != {len(prediction_results)}"
        )

    evaluation_records: list[dict[str, Any]] = []

    for index, (example, prediction) in enumerate(zip(test_dataset, prediction_results, strict=True)):
        text = example.get("text", "")

        prompt = example.get("query")
        if not isinstance(prompt, str) or not prompt.strip():
            user_matches = _USER_PATTERN.findall(text)
            if not user_matches:
                raise ValueError(f"test_dataset[{index}] has no query or <user> block")
            prompt = user_matches[0]

        answers = example.get("answers")
        ground_truth_from_text = answers is None
        if answers is None:
            answers = text

        prediction_parse_error = None
        try:
            predicted_tool_calls = _parse_tool_calls(
                prediction,
                source=f"prediction_results[{index}]",
                require_tags=True,
            )
        except ValueError as exc:
            predicted_tool_calls = []
            prediction_parse_error = str(exc)

        actual_tool_calls = _parse_tool_calls(
            answers,
            source=f"test_dataset[{index}] ground truth",
            require_tags=ground_truth_from_text,
        )
        comparison = compare_tool_calls(
            predicted_tool_calls,
            actual_tool_calls,
            prediction_parse_error=prediction_parse_error,
        )

        evaluation_records.append({
            "user_prompt": prompt.strip(),
            "predicted_tool_calls": predicted_tool_calls,
            "actual_tool_calls": actual_tool_calls,
            **comparison,
            "prediction_parse_error": prediction_parse_error,
            "predicted_raw": prediction if prediction_parse_error else None,
        })

    return evaluation_records


def save_accuracy_summary(
    accuracy_metrics: Mapping[str, int | float],
    output_path: str | Path = "tool_call_eval_accuracy_qwen-3-0.6B.json",
) -> Path:
    """Write aggregate function-call accuracy metrics to a JSON file."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as summary_file:
        json.dump(dict(accuracy_metrics), summary_file, ensure_ascii=False, indent=2)
    print(f"Saved accuracy summary to {output_path.resolve()}")
    return output_path


def save_function_call_evaluation(
    evaluation_records: Sequence[Mapping[str, Any]],
    output_stem: str | Path = "tool_call_eval_results_qwen-3-0.6B",
) -> dict[str, Path]:
    """Write evaluation records to CSV and lossless JSON files."""
    output_stem = Path(output_stem)
    output_stem.parent.mkdir(parents=True, exist_ok=True)
    csv_path = output_stem.parent / f"{output_stem.name}.csv"
    json_path = output_stem.parent / f"{output_stem.name}.json"

    normalized_records = [dict(record) for record in evaluation_records]
    required_fields = {
        "user_prompt",
        "predicted_tool_calls",
        "actual_tool_calls",
        "tool_calls_match",
        "arguments_match",
        "exact_match",
    }
    for index, record in enumerate(normalized_records):
        missing_fields = required_fields - record.keys()
        if missing_fields:
            raise ValueError(
                f"evaluation_records[{index}] is missing fields: "
                f"{sorted(missing_fields)}"
            )

    with csv_path.open("w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "user_prompt",
                "predicted_tool_calls",
                "actual_tool_calls",
                "tool_calls_match",
                "arguments_match",
                "exact_match",
                "prediction_parse_error",
                "predicted_raw",
            ],
        )
        writer.writeheader()
        for record in normalized_records:
            writer.writerow({
                "user_prompt": record["user_prompt"].replace(
                    "\r", "\\r"
                ).replace("\n", "\\n"),
                "predicted_tool_calls": json.dumps(
                    record["predicted_tool_calls"], ensure_ascii=False
                ),
                "actual_tool_calls": json.dumps(
                    record["actual_tool_calls"], ensure_ascii=False
                ),
                "tool_calls_match": record["tool_calls_match"],
                "arguments_match": record["arguments_match"],
                "exact_match": record["exact_match"],
                "prediction_parse_error": (
                    record.get("prediction_parse_error") or ""
                ).replace("\r", "\\r").replace("\n", "\\n"),
                "predicted_raw": json.dumps(
                    record["predicted_raw"], ensure_ascii=False
                ) if record.get("predicted_raw") is not None else "",
            })

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(normalized_records, json_file, ensure_ascii=False, indent=2)

    print(f"Saved {len(normalized_records):,} records to {csv_path.resolve()}")
    print(f"Saved {len(normalized_records):,} records to {json_path.resolve()}")
    return {"csv": csv_path, "json": json_path}


# Use the same full dataset passed to test_function_calling_examples.
evaluation_results = evaluate_function_call_predictions(test_dataset, results)
accuracy_metrics = calculate_function_call_accuracy(evaluation_results)
print(json.dumps(accuracy_metrics, indent=2))
accuracy_summary_path = save_accuracy_summary(accuracy_metrics)
evaluation_files = save_function_call_evaluation(evaluation_results)
evaluation_results[:2]
